# Hands-On 7: Build Document Research Crew

### Definition
**RAG (Retrieval-Augmented Generation)** retrieves relevant document content and gives it to an agent so the response is grounded in the knowledge base.

### Flow
`Documents → Split → Embeddings → FAISS → Retrieval Tool → Agent → Grounded Response + Sources`

In [3]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import os
from dotenv import load_dotenv

load_dotenv(r"D:\Agent_Flow\.env", override=True)

print("OpenRouter key loaded:", bool(os.getenv("OPENROUTER_API_KEY")))
print("Model:", os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini"))
print("Base URL:", os.getenv("OPENAI_BASE_URL"))

OpenRouter key loaded: True
Model: openai/gpt-4o-mini
Base URL: https://openrouter.ai/api/v1


In [14]:
llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini"),
    temperature=0,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")
)

print("LLM and embeddings configured")

LLM and embeddings configured


## 1. Load Documents

For this hands-on, we create small sample documents. In a real project, these can come from PDFs, Word files, web pages, or databases.

In [15]:
documents = [
    Document(
        page_content=(
            "LangGraph is a framework for building stateful agent workflows. "
            "It supports graph-based orchestration, persistence, streaming, "
            "and human-in-the-loop workflows."
        ),
        metadata={"source": "langgraph_notes.txt", "topic": "frameworks"}
    ),
    Document(
        page_content=(
            "CrewAI is a framework for orchestrating multi-agent systems. "
            "Agents have roles, goals, and backstories and can work together "
            "through sequential or hierarchical processes."
        ),
        metadata={"source": "crewai_notes.txt", "topic": "frameworks"}
    ),
    Document(
        page_content=(
            "FAISS is a vector similarity search library commonly used for "
            "local retrieval and prototyping. It stores vector representations "
            "and retrieves documents similar to a query."
        ),
        metadata={"source": "vector_db_notes.txt", "topic": "vector-database"}
    ),
    Document(
        page_content=(
            "Embeddings are numerical representations of text that capture "
            "semantic meaning. Similar text produces vectors that are close "
            "to each other in vector space."
        ),
        metadata={"source": "embedding_notes.txt", "topic": "embeddings"}
    ),
    Document(
        page_content=(
            "RAG combines retrieval with generation. A retriever finds relevant "
            "document chunks and the language model uses those chunks to produce "
            "a grounded response."
        ),
        metadata={"source": "rag_notes.txt", "topic": "rag"}
    )
]

print(f"Loaded {len(documents)} documents")

Loaded 5 documents


## 2. Split Document Content

Chunking breaks long documents into smaller pieces so retrieval can return focused context.

In [16]:
def split_documents(docs, chunk_size=180, overlap=30):
    chunks = []

    for doc in docs:
        text = doc.page_content

        start = 0
        chunk_number = 1

        while start < len(text):
            end = min(start + chunk_size, len(text))
            chunk = text[start:end].strip()

            if chunk:
                metadata = dict(doc.metadata)
                metadata["chunk"] = chunk_number
                chunks.append(
                    Document(
                        page_content=chunk,
                        metadata=metadata
                    )
                )

            if end >= len(text):
                break

            start = end - overlap
            chunk_number += 1

    return chunks


chunks = split_documents(documents)

print("Original documents:", len(documents))
print("Chunks:", len(chunks))

for i, chunk in enumerate(chunks[:5], 1):
    print(f"{i}. {chunk.metadata} -> {chunk.page_content[:80]}...")

Original documents: 5
Chunks: 5
1. {'source': 'langgraph_notes.txt', 'topic': 'frameworks', 'chunk': 1} -> LangGraph is a framework for building stateful agent workflows. It supports grap...
2. {'source': 'crewai_notes.txt', 'topic': 'frameworks', 'chunk': 1} -> CrewAI is a framework for orchestrating multi-agent systems. Agents have roles, ...
3. {'source': 'vector_db_notes.txt', 'topic': 'vector-database', 'chunk': 1} -> FAISS is a vector similarity search library commonly used for local retrieval an...
4. {'source': 'embedding_notes.txt', 'topic': 'embeddings', 'chunk': 1} -> Embeddings are numerical representations of text that capture semantic meaning. ...
5. {'source': 'rag_notes.txt', 'topic': 'rag', 'chunk': 1} -> RAG combines retrieval with generation. A retriever finds relevant document chun...


## 3. Generate Embeddings and Store Vectors

Each chunk is converted into an embedding vector and stored in FAISS for similarity search.

In [17]:
vector_store = FAISS.from_documents(chunks, embeddings)

print("FAISS vector store created")
print("Total vectors:", vector_store.index.ntotal)

FAISS vector store created
Total vectors: 5


In [18]:
query = "What is RAG and how does retrieval work?"

results = vector_store.similarity_search(query, k=3)

print("Query:", query)
print("\nRetrieved documents:")

for i, doc in enumerate(results, 1):
    print(f"\nSource {i}: {doc.metadata['source']}")
    print(doc.page_content)

Query: What is RAG and how does retrieval work?

Retrieved documents:

Source 1: rag_notes.txt
RAG combines retrieval with generation. A retriever finds relevant document chunks and the language model uses those chunks to produce a grounded response.

Source 2: vector_db_notes.txt
FAISS is a vector similarity search library commonly used for local retrieval and prototyping. It stores vector representations and retrieves documents similar to a query.

Source 3: langgraph_notes.txt
LangGraph is a framework for building stateful agent workflows. It supports graph-based orchestration, persistence, streaming, and human-in-the-loop workflows.


## 4. Create Retrieval Tool

The tool searches the vector store and returns both the retrieved content and source metadata.

In [19]:
@tool
def retrieve_documents(query: str) -> str:
    """Retrieve relevant document chunks and their sources."""
    try:
        docs = vector_store.similarity_search(query, k=3)

        if not docs:
            return "No relevant documents found."

        results = []

        for i, doc in enumerate(docs, 1):
            results.append(
                f"Source {i}: {doc.metadata.get('source', 'unknown')} "
                f"(topic={doc.metadata.get('topic', 'unknown')}, "
                f"chunk={doc.metadata.get('chunk', 'unknown')})\n"
                f"{doc.page_content}"
            )

        return "\n\n---\n\n".join(results)

    except Exception as e:
        return f"Retrieval error: {e}"


print("Retrieval tool created:", retrieve_documents.name)

Retrieval tool created: retrieve_documents


In [20]:
print(retrieve_documents.run("What are the main features of CrewAI?"))

Source 1: crewai_notes.txt (topic=frameworks, chunk=1)
CrewAI is a framework for orchestrating multi-agent systems. Agents have roles, goals, and backstories and can work together through sequential or hierarchical processes.

---

Source 2: langgraph_notes.txt (topic=frameworks, chunk=1)
LangGraph is a framework for building stateful agent workflows. It supports graph-based orchestration, persistence, streaming, and human-in-the-loop workflows.

---

Source 3: vector_db_notes.txt (topic=vector-database, chunk=1)
FAISS is a vector similarity search library commonly used for local retrieval and prototyping. It stores vector representations and retrieves documents similar to a query.


## 5. Create RAG Agent

The agent is instructed to use retrieved information and avoid unsupported claims.

In [24]:
import os

llm = os.getenv(
    "OPENAI_MODEL_NAME",
    "openai/gpt-4o-mini"
)

print("CrewAI LLM:", llm)

CrewAI LLM: openai/gpt-4o-mini


In [25]:
rag_agent = Agent(
    role="Document Research Specialist",

    goal=(
        "Answer questions accurately using retrieved document content "
        "and clearly identify the supporting sources."
    ),

    backstory=(
        "You are a meticulous research specialist. You always retrieve "
        "relevant documents before answering, stay grounded in the source "
        "material, and clearly mention the source names."
    ),

    llm=llm,
    tools=[retrieve_documents],
    verbose=True,
    allow_delegation=False
)

print("RAG agent created")

RAG agent created


## 6. Create Research Task

In [26]:
research_task = Task(
    description=(
        "Answer: What is RAG, how does it work, and why are embeddings "
        "and FAISS useful in a RAG pipeline? Use the retrieval tool. "
        "Ground the answer only in retrieved information and include source names."
    ),
    expected_output=(
        "A concise grounded explanation of RAG, embeddings, and FAISS "
        "with source names."
    ),
    agent=rag_agent
)

print("Research task created")

Research task created


## 7. Execute Document Research Crew

In [27]:
rag_crew = Crew(
    agents=[rag_agent],
    tasks=[research_task],
    process=Process.sequential,
    verbose=True,
    memory=False
)

print("=" * 60)
print("EXECUTING DOCUMENT RESEARCH CREW")
print("=" * 60)

rag_result = await rag_crew.kickoff_async()

print("\n" + "=" * 60)
print("GROUNDED RESPONSE")
print("=" * 60)
print(rag_result)

EXECUTING DOCUMENT RESEARCH CREW


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 406a6b0b-9d94-40b6-9c45-fe45547ff80a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer: What is RAG, how does it work, and why are embeddings and FAISS useful in a RAG pipeline? Use    │
│  the retrieval tool. Ground the answer only in retrieved information and include source names.                  │
│  ID: be38b576-d49d-495d-892f-9273de41675d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Document Research Specialist                                                                            │
│                                                                                                                 │
│  Task: Answer: What is RAG, how does it work, and why are embeddings and FAISS useful in a RAG pipeline? Use    │
│  the retrieval tool. Ground the answer only in retrieved information and include source names.                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: retrieve_documents                                                                                       │
│  Args: {'query': 'RAG embeddings FAISS'}                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool retrieve_documents executed with result: Source 1: vector_db_notes.txt (topic=vector-database, chunk=1)
FAISS is a vector similarity search library commonly used for local retrieval and prototyping. It stores vector representations and retri...


... (3 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer: What is RAG, how does it work, and why are embeddings and FAISS useful in a RAG pipeline? Use    │
│  the retrieval tool. Ground the answer only in retrieved information and include source names.                  │
│  Agent: Document Research Specialist                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 406a6b0b-9d94-40b6-9c45-fe45547ff80a                                                                       │
│  Final Output: RAG, or Retrieval-Augmented Generation, combines retrieval with generation by having a           │
│  retriever identify relevant document chunks, which are then used by the language model to generate a grounded  │
│  response. This dual approach allows for more accurate and contextually relevant answers.                       │
│                                                                                                                 │
│  Embeddings are numerical representations of text that capture their semantic meaning. Texts that are similar   │
│  produce vectors that are close to each other in the vector space, facilitating effective similarity searches.  │
│                                                                                                                 │
│  FAISS, the Facebook AI Similarity Search library, is utilized for vector similarity searches. It stores these  │
│  vector representations and retrieves documents that are similar to a given query, making it an essential tool  │
│  for efficient retrieval in RAG pipelines.                                                                      │
│                                                                                                                 │
│  **Sources:**                                                                                                   │
│  - rag_notes.txt                                                                                                │
│  - embedding_notes.txt                                                                                          │
│  - vector_db_notes.txt                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


GROUNDED RESPONSE
RAG, or Retrieval-Augmented Generation, combines retrieval with generation by having a retriever identify relevant document chunks, which are then used by the language model to generate a grounded response. This dual approach allows for more accurate and contextually relevant answers.

Embeddings are numerical representations of text that capture their semantic meaning. Texts that are similar produce vectors that are close to each other in the vector space, facilitating effective similarity searches.

FAISS, the Facebook AI Similarity Search library, is utilized for vector similarity searches. It stores these vector representations and retrieves documents that are similar to a given query, making it an essential tool for efficient retrieval in RAG pipelines.

**Sources:**
- rag_notes.txt
- embedding_notes.txt
- vector_db_notes.txt


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 8. Display Retrieved Sources

The retrieval layer can independently show exactly which chunks supported the answer.

In [28]:
source_query = "What is RAG, embeddings, and FAISS?"

source_docs = vector_store.similarity_search(source_query, k=3)

print("=" * 60)
print("SOURCES")
print("=" * 60)

for i, doc in enumerate(source_docs, 1):
    print(f"\n[{i}] {doc.metadata.get('source')}")
    print(f"Topic: {doc.metadata.get('topic')}")
    print(f"Chunk: {doc.metadata.get('chunk')}")
    print(f"Content: {doc.page_content}")

SOURCES

[1] vector_db_notes.txt
Topic: vector-database
Chunk: 1
Content: FAISS is a vector similarity search library commonly used for local retrieval and prototyping. It stores vector representations and retrieves documents similar to a query.

[2] rag_notes.txt
Topic: rag
Chunk: 1
Content: RAG combines retrieval with generation. A retriever finds relevant document chunks and the language model uses those chunks to produce a grounded response.

[3] crewai_notes.txt
Topic: frameworks
Chunk: 1
Content: CrewAI is a framework for orchestrating multi-agent systems. Agents have roles, goals, and backstories and can work together through sequential or hierarchical processes.


## 9. Test Out-of-Scope Query

A grounded RAG system should not invent information that is absent from its knowledge base.

In [29]:
out_of_scope_task = Task(
    description=(
        "Answer: What is quantum computing? Use the retrieval tool. "
        "If the knowledge base does not contain the answer, explicitly say "
        "that the available documents do not provide enough information."
    ),
    expected_output="An honest grounded response with no unsupported claims.",
    agent=rag_agent
)

oos_crew = Crew(
    agents=[rag_agent],
    tasks=[out_of_scope_task],
    process=Process.sequential,
    verbose=True,
    memory=False
)

oos_result = await oos_crew.kickoff_async()

print("\nOUT-OF-SCOPE RESPONSE")
print(oos_result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d6acbc46-0585-4529-9a81-1df96ffc8d7f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer: What is quantum computing? Use the retrieval tool. If the knowledge base does not contain the    │
│  answer, explicitly say that the available documents do not provide enough information.                         │
│  ID: daf0e1d6-5141-4e88-a755-d681e6378936                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Document Research Specialist                                                                            │
│                                                                                                                 │
│  Task: Answer: What is quantum computing? Use the retrieval tool. If the knowledge base does not contain the    │
│  answer, explicitly say that the available documents do not provide enough information.                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: retrieve_documents                                                                                       │
│  Args: {'query': 'What is quantum computing?'}                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool retrieve_documents executed with result: Source 1: rag_notes.txt (topic=rag, chunk=1)
RAG combines retrieval with generation. A retriever finds relevant document chunks and the language model uses those chunks to produce a grounded response....


╭─────────────────────────────────────── ✅ Tool Execution Completed (#2) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: retrieve_documents                                                                                       │
│  Output: Source 1: rag_notes.txt (topic=rag, chunk=1)                                                           │
│  RAG combines retrieval with generation. A retriever finds relevant document chunks and the language model      │
│  uses those chunks to produce a grounded response.                                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  Source 2: crewai_notes.txt (topic=frameworks, chunk=1)                                                         │
│  CrewAI is a framework for orchestrating multi-agent systems. Agents have roles, goals, and backstories and     │
│  can work together through sequential or hierarchical processes.                                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  Source 3: embedding_notes.txt (topic=embeddings, chunk=1)                                                      │
│  Embeddings are numerical representations of text that capture semantic meaning. Similar text produces vectors  │
│  that are close to each other in vector space.                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (2 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer: What is quantum computing? Use the retrieval tool. If the knowledge base does not contain the    │
│  answer, explicitly say that the available documents do not provide enough information.                         │
│  Agent: Document Research Specialist                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d6acbc46-0585-4529-9a81-1df96ffc8d7f                                                                       │
│  Final Output: The available documents do not provide enough information on quantum computing.                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


OUT-OF-SCOPE RESPONSE
The available documents do not provide enough information on quantum computing.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 10. Final Verification

- ✅ Documents loaded
- ✅ Content split into chunks
- ✅ Embeddings generated
- ✅ Vectors stored in FAISS
- ✅ Retrieval tool created
- ✅ Retriever assigned to agent
- ✅ Grounded response generated
- ✅ Sources displayed
- ✅ Out-of-scope query tested

### Final RAG flow

`Documents → Chunks → Embeddings → FAISS → Retriever → Agent → Grounded Answer → Sources`